In [1]:
# data ingestion

# Phase 1 – Document Ingestion

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "3D_MedDiffusion_A_3D_Medical_Latent_Diffusion_Model_for_Controllable_and_High-Quality_Medical_Image_Generation.pdf"

loader = PyPDFLoader(file_path)

docs = loader.load()

print("Total pages loaded:", len(docs))

/home/anjucv/.cache/pypoetry/virtualenvs/rag-task1-_QzYQtjV-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total pages loaded: 13


In [60]:
docs[0].metadata

{'producer': 'pdfTeX-1.40.24; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2025-11-26T12:46:04+05:30',
 'moddate': '2025-12-05T21:23:14-05:00',
 'ieee article id': '11063450',
 'trapped': 'False',
 'ieee issue id': '11279972',
 'subject': 'IEEE Transactions on Medical Imaging;2025;44;12;10.1109/TMI.2025.3585372',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4',
 'ieee publication id': '42',
 'title': '3D MedDiffusion: A 3D Medical Latent Diffusion Model for Controllable and High-Quality Medical Image Generation',
 'source': '3D_MedDiffusion_A_3D_Medical_Latent_Diffusion_Model_for_Controllable_and_High-Quality_Medical_Image_Generation.pdf',
 'total_pages': 13,
 'page': 0,
 'page_label': '4960'}

In [61]:
print(docs[1].page_content)

WANG et al.: 3D MedDiffusion: A 3D MEDICAL LATENT DIFFUSION MODEL 4961
limited in both resolution and quality [14], [15]. Some works
rely on slice-wise 2D generation [16], [17], [18], while clinical
practice needs 3D volumetric imaging. Second, the lack of
efficient and controllable generation mechanisms forces the
development of task-specific models, thereby increasing devel-
opment costs.
We propose 3D Medical Latent Diffusion (3D MedDif-
fusion), a large-scale 3D generative model designed for
high-fidelity generation of medical images. First, 3D Med-
Diffusion introduces a novel Patch-V olume Autoencoder that
compresses high-resolution images into low-resolution latent
representations via a patch-wise encoder and recovers the
entire image with a volume-wise decoder. Patch-wise encoding
reduces the computational overhead in the training stage
by exploiting the local properties in 3D medical images,
while volume-wise decoding ensures artifact-free reconstruc-
tion results. Second, we 

In [3]:
#chunking

# Phase 2 – Text Chunking

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)

texts = text_splitter.split_documents(docs)

# Remove empty chunks
texts = [chunk for chunk in texts if chunk.page_content and chunk.page_content.strip()]

print("Total chunks:", len(texts))

Total chunks: 154


In [62]:
for i, chunk in enumerate(texts[:5]):
    print(f"\n---- CHUNK {i} ----\n")
    print(chunk.page_content)


---- CHUNK 0 ----

4960 IEEE TRANSACTIONS ON MEDICAL IMAGING, VOL. 44, NO. 12, DECEMBER 2025
3D MedDiffusion: A 3D Medical Latent Diffusion
Model for Controllable and High-Quality
Medical Image Generation
Haoshen Wang
 , Zhentao Liu
 , Kaicong Sun
 , Xiaodong Wang,
Dinggang Shen
 , Fellow, IEEE, and Zhiming Cui
Abstract— The generation of medical images presents
significant challenges due to their high-resolution and
three-dimensional nature. Existing methods often yield sub-

---- CHUNK 1 ----

three-dimensional nature. Existing methods often yield sub-
optimal performance in generating high-quality 3D medical
images, and there is currently no universal generative
framework for medical imaging. In this paper, we introduce
a 3D Medical Latent Diffusion (3D MedDiffusion) model for
controllable, high-quality 3D medical image generation. 3D
MedDiffusion incorporates a novel, highly efficient Patch-
Volume Autoencoder that compresses medical images into

---- CHUNK 2 ----

Volume Autoenco

In [5]:
# create documents with meta data

# Phase 3 – Embedding Generation

In [6]:
documents = []

for i, chunk in enumerate(texts):

    metadata = chunk.metadata.copy()
    metadata["chunk_index"] = i

    documents.append({
        "text": chunk.page_content,
        "metadata": metadata
    })

print("Documents prepared:", len(documents))

Documents prepared: 154


In [7]:
# embedding and vector storage

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1175.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# qdrant vector store

# Phase 4 – Vector Storage (Qdrant)

In [10]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient(":memory:")

collection_name = "medical_paper"

vector_size = len(embeddings.embed_query("test"))

if not client.collection_exists(collection_name):

    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=vector_size,
            distance=Distance.COSINE
        )
    )

print("Collection created")

Collection created


In [11]:
# store embedding in quadrent

In [12]:
from qdrant_client.models import PointStruct
import uuid

points = []

for doc in documents:

    vector = embeddings.embed_query(doc["text"])

    payload = {
        "text": doc["text"],
        "metadata": doc["metadata"]
    }

    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=vector,
            payload=payload
        )
    )

client.upsert(
    collection_name=collection_name,
    points=points
)

print("Embeddings stored:", len(points))

Embeddings stored: 154


In [13]:
collection_info = client.get_collection(collection_name)

print("Total vectors:", collection_info.points_count)
print("Vector dimension:", collection_info.config.params.vectors.size)
print("Distance metric:", collection_info.config.params.vectors.distance)

Total vectors: 154
Vector dimension: 384
Distance metric: Cosine


In [ ]:
# query = "What is 3D MedDiffusion?"

# query_vector = embeddings.embed_query(query)

# results = client.query_points(
#     collection_name=collection_name,
#     query=query_vector,
#     limit=3
# )

# print("Top Results:\n")

# for r in results.points:

#     print("Score:", r.score)
#     print(r.payload["text"][:300])
#     print("Metadata:", r.payload["metadata"])
#     print("-----")

Top Results:

Score: 0.6571710753792428
WANG et al.: 3D MedDiffusion: A 3D MEDICAL LATENT DIFFUSION MODEL 4961
limited in both resolution and quality [14], [15]. Some works
rely on slice-wise 2D generation [16], [17], [18], while clinical
practice needs 3D volumetric imaging. Second, the lack of
efficient and controllable generation mecha
Metadata: {'producer': 'pdfTeX-1.40.24; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-11-26T12:46:04+05:30', 'moddate': '2025-12-05T21:23:14-05:00', 'ieee article id': '11063450', 'trapped': 'False', 'ieee issue id': '11279972', 'subject': 'IEEE Transactions on Medical Imaging;2025;44;12;10.1109/TMI.2025.3585372', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'ieee publication id': '42', 'title': '3D MedDiffusion: A 3D Medical Latent Diffusion Model for Controllable and High-Quality Medical Image Generati

# Phase 5 – Retrieval Using LlamaIndex

In [15]:
# connect llamaindex

In [19]:
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Disable LLM
Settings.llm = None

# Set embedding model (IMPORTANT)
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Connect to Qdrant
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

# Create index
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    storage_context=storage_context
)

print("Vector index created successfully")

LLM is explicitly disabled. Using MockLLM.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1070.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector index created successfully


In [53]:
query_engine = index.as_query_engine(similarity_top_k=3)

response = query_engine.query("What is 3D diffusion model")

for node in response.source_nodes:
    print("Score:", node.score)
    print(node.node.text[:300])
    print("------")

Score: 0.6552408551028988
extractor, we randomly select three orthogonal 2D planes from
input data {pn}3
n=1 ⊂ x and reconstruct the planes { ˜pn}3
n=1 ⊂ ˜x
at the same position to compute perceptural loss:
LT P =
3∑
n=1
∥φ(p n) − φ( ˜pn)∥2, (5)
where φ is the pre-trained VGG-16 network [38], [39].
B. 3D Medical Diffusion Mo
------
Score: 0.6522311162342799
WANG et al.: 3D MedDiffusion: A 3D MEDICAL LATENT DIFFUSION MODEL 4961
limited in both resolution and quality [14], [15]. Some works
rely on slice-wise 2D generation [16], [17], [18], while clinical
practice needs 3D volumetric imaging. Second, the lack of
efficient and controllable generation mecha
------
Score: 0.6135437713232879
4960 IEEE TRANSACTIONS ON MEDICAL IMAGING, VOL. 44, NO. 12, DECEMBER 2025
3D MedDiffusion: A 3D Medical Latent Diffusion
Model for Controllable and High-Quality
Medical Image Generation
Haoshen Wang
 , Zhentao Liu
 , Kaicong Sun
 , Xiaodong Wang,
Dinggang Shen
 , Fellow, IEEE, and Zhiming Cui
Abstra
-----

In [54]:
context_list = []

for i, node in enumerate(response.source_nodes):
    text = node.node.text
    context_list.append(text)
context = "\n\n".join(context_list)
context

'extractor, we randomly select three orthogonal 2D planes from\ninput data {pn}3\nn=1 ⊂ x and reconstruct the planes { ˜pn}3\nn=1 ⊂ ˜x\nat the same position to compute perceptural loss:\nLT P =\n3∑\nn=1\n∥φ(p n) − φ( ˜pn)∥2, (5)\nwhere φ is the pre-trained VGG-16 network [38], [39].\nB. 3D Medical Diffusion Model\n1) Diffusion Model: The diffusion model in 3D MedDiffu-\nsion operates in the compact latent space, either patch-wise\n(xi) or volume-wise ( X). In this section, we use x to denote\n\nWANG et al.: 3D MedDiffusion: A 3D MEDICAL LATENT DIFFUSION MODEL 4961\nlimited in both resolution and quality [14], [15]. Some works\nrely on slice-wise 2D generation [16], [17], [18], while clinical\npractice needs 3D volumetric imaging. Second, the lack of\nefficient and controllable generation mechanisms forces the\ndevelopment of task-specific models, thereby increasing devel-\nopment costs.\nWe propose 3D Medical Latent Diffusion (3D MedDif-\nfusion), a large-scale 3D generative model desi

In [25]:
query = "What is BiFlownet"

In [55]:
print(context)

extractor, we randomly select three orthogonal 2D planes from
input data {pn}3
n=1 ⊂ x and reconstruct the planes { ˜pn}3
n=1 ⊂ ˜x
at the same position to compute perceptural loss:
LT P =
3∑
n=1
∥φ(p n) − φ( ˜pn)∥2, (5)
where φ is the pre-trained VGG-16 network [38], [39].
B. 3D Medical Diffusion Model
1) Diffusion Model: The diffusion model in 3D MedDiffu-
sion operates in the compact latent space, either patch-wise
(xi) or volume-wise ( X). In this section, we use x to denote

WANG et al.: 3D MedDiffusion: A 3D MEDICAL LATENT DIFFUSION MODEL 4961
limited in both resolution and quality [14], [15]. Some works
rely on slice-wise 2D generation [16], [17], [18], while clinical
practice needs 3D volumetric imaging. Second, the lack of
efficient and controllable generation mechanisms forces the
development of task-specific models, thereby increasing devel-
opment costs.
We propose 3D Medical Latent Diffusion (3D MedDif-
fusion), a large-scale 3D generative model designed for

4960 IEEE TRAN

# Phase 6 – Response Generation Using Gemini

In [ ]:
prompt = f"""
You are a helpful AI assistant.

Answer the question using ONLY the provided context. give a detailed answer.
If the answer is not in the context, say "Answer not found in context".

Context:
{context}

Question:
{query}

Answer:
"""

In [42]:
import os

In [27]:
from dotenv import load_dotenv
load_dotenv()

True

In [57]:
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

response_llm = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

In [44]:
response_llm

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='BiFlowNet is a novel noise estimator that integrates both intra-patch flow and inter-patch flow to generate latent representations. It is described as being perfectly compatible with the proposed Patch-Volume Autoencoder.'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='V3uqaYPiB5uGmNMPzrGM-A0',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=41,
    prompt_token_count=430,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=430
      ),
    ],
    thoughts_token_count=212,
    total_token_count=683
  )
)

In [58]:
response_llm

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='The diffusion model in 3D MedDiffusion operates in the compact latent space, either patch-wise (xi) or volume-wise (X). 3D MedDiffusion is presented as a large-scale 3D generative model designed for controllable and high-quality medical image generation.'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='R3yqaf7GF8W1juMPhqKQ2Qk',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=58,
    prompt_token_count=498,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=498
      ),
    ],
    thoughts_token_count=554,
    total_token_cou

In [46]:
response_llm.candidates[0].content.parts[0].text

'BiFlowNet is a novel noise estimator that integrates both intra-patch flow and inter-patch flow to generate latent representations. It is described as being perfectly compatible with the proposed Patch-Volume Autoencoder.'

In [47]:
query = "What is 3D diffusion model?"

In [59]:
print(response_llm.candidates[0].content.parts[0].text)

The diffusion model in 3D MedDiffusion operates in the compact latent space, either patch-wise (xi) or volume-wise (X). 3D MedDiffusion is presented as a large-scale 3D generative model designed for controllable and high-quality medical image generation.
